# Independent DPE methods on saved backbones
Enable Internet and optionally two GPUs. Attach the locally built `dpe-independent-code.zip` as dataset `dpe-independent-code`. Every declared configuration receives a test result or an explicit failure; no candidate is selected. Raw features and saved heads are recovered from the two public source notebooks. No backbone is trained. Preserve the entire `dpe_independent_study` output for resumption.

In [ ]:
from pathlib import Path, PurePosixPath
import sys, zipfile
CODE_BUNDLE = Path('/kaggle/input/dpe-independent-code/dpe-independent-code.zip')
CODE = Path('/kaggle/working/dpe_independent_code')
WORK = Path('/kaggle/working/dpe_independent_study')
ARTIFACTS = Path('/kaggle/working/dpe_saved_artifacts')
with zipfile.ZipFile(CODE_BUNDLE) as archive:
    for name in archive.namelist():
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name:
            raise ValueError('Unsafe code bundle path')
    archive.extractall(CODE)
sys.path.insert(0, str(CODE))
import os
os.chdir(CODE)
import torch
from src.methods.dpe_independent.recovery import recover
from src.methods.dpe_independent.sources import discover
from src.methods.dpe_independent.storage import atomic_json
from src.methods.dpe_independent.launcher import launch


In [ ]:
recovered = recover(ARTIFACTS)
sources, blockers = discover([ARTIFACTS])
SOURCE_MANIFEST = WORK / 'sources.json'
atomic_json(SOURCE_MANIFEST, dict(sources=sources, blockers=blockers))
print('Sources:', [source['id'] for source in sources])
print('Artifact blockers:', blockers)
if len(sources) != 7:
    raise RuntimeError('Expected all seven saved sources; inspect blockers before running')


In [ ]:
DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())] or ['cpu']
completion = launch(SOURCE_MANIFEST, WORK / 'results', devices=DEVICES)
print(completion)
print('All rows:', WORK / 'results/per_configuration.csv')
print('Recipe/allocation summaries:', WORK / 'results/summary.csv')
